# DSA 210 — Personalized Well-being Prediction from Sleep & Lifestyle Data
**Author:** Deniz Elbek | **Final Submission — Machine Learning Models**

Full dataset: 44 nights (April 1 – May 14, 2026), 37 complete observations for ML.  
Features: 24 (sleep + physiological + lifestyle + weather + circadian).  
Targets: morning feel rating (1–10) and daily productivity rating (1–10).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
import xgboost as xgb
import shap

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/sleep_data.csv', parse_dates=['date'])
ml_df = df[df[['duration_h','morning_feel','productivity']].notna().all(axis=1)].copy().reset_index(drop=True)
print(f'ML dataset: {len(ml_df)} complete observations × {df.shape[1]} cols')

## 2. Feature Preparation

In [ ]:
FEATURES = [
    'duration_h','deep_min','rem_min','core_min','awake_min',
    'sleep_efficiency','apple_sleep_score','nap_duration_min',
    'hrv_ms','spo2_pct','resting_hr','resp_rate',
    'steps','active_cal','exercise_min',
    'caffeine_mg','last_caffeine_hour','alcohol_scale','last_meal_hour',
    'temp_c','humidity_pct','pressure_hpa','daylight_h','bedtime_after_sunset'
]
FEAT_LABELS = [f.replace('_',' ').title() for f in FEATURES]

X_raw = ml_df[FEATURES]
y_feel = ml_df['morning_feel'].values
y_prod = ml_df['productivity'].values

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X_raw), columns=FEATURES)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

print(f'Features: {len(FEATURES)}')
print(f'Imputed NaNs per column:')
print(X_raw.isnull().sum()[X_raw.isnull().sum()>0])

## 3. Baseline: Naive Mean Predictor

In [ ]:
# Establish baseline — predict the training mean for each LOO fold
loo = LeaveOneOut()
baseline_feel = []; baseline_prod = []
for train_idx, test_idx in loo.split(X_scaled):
    baseline_feel.append(y_feel[train_idx].mean())
    baseline_prod.append(y_prod[train_idx].mean())

base_mae_feel = mean_absolute_error(y_feel, baseline_feel)
base_mae_prod = mean_absolute_error(y_prod, baseline_prod)
base_r2_feel  = r2_score(y_feel, baseline_feel)
base_r2_prod  = r2_score(y_prod, baseline_prod)

print('=== NAIVE BASELINE (predict training mean) ===')
print(f'Morning Feel — MAE: {base_mae_feel:.3f}, R²: {base_r2_feel:.3f}')
print(f'Productivity  — MAE: {base_mae_prod:.3f}, R²: {base_r2_prod:.3f}')

## 4. Model Training & LOOCV Evaluation

In [ ]:
models = {
    'Ridge':        Ridge(alpha=1.0),
    'Lasso':        Lasso(alpha=0.1, max_iter=10000),
    'ElasticNet':   ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=3,
                                          min_samples_leaf=3, random_state=42),
    'XGBoost':      xgb.XGBRegressor(n_estimators=50, max_depth=2,
                                     learning_rate=0.1, subsample=0.8,
                                     random_state=42, verbosity=0),
}
PRIMARY = ['Ridge','Lasso','ElasticNet']

results = {}; predictions = {}
for tname, y in [('morning_feel', y_feel), ('productivity', y_prod)]:
    results[tname] = {}; predictions[tname] = {}
    for name, model in models.items():
        X = X_scaled if name in PRIMARY else X_imp.values
        preds = cross_val_predict(model, X, y, cv=loo)
        results[tname][name] = {
            'MAE':  round(mean_absolute_error(y, preds), 3),
            'RMSE': round(np.sqrt(mean_squared_error(y, preds)), 3),
            'R2':   round(r2_score(y, preds), 3),
            'vs_baseline_MAE': round(mean_absolute_error(y, preds) - (base_mae_feel if 'feel' in tname else base_mae_prod), 3)
        }
        predictions[tname][name] = preds

for tname in ['morning_feel','productivity']:
    print(f'\n=== {tname.upper().replace("_"," ")} ===')
    res_df = pd.DataFrame(results[tname]).T
    res_df.index.name = 'Model'
    print(res_df.to_string())

## 5. Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18,8))
fig.suptitle('LOOCV Predicted vs Actual (n=37)', fontsize=13, fontweight='bold')
colors = ['steelblue','coral','mediumpurple','seagreen','darkorange']

for row, (tname, y) in enumerate([('morning_feel',y_feel),('productivity',y_prod)]):
    for col, (name, color) in enumerate(zip(models.keys(), colors)):
        ax = axes[row][col]
        preds = predictions[tname][name]
        r2 = results[tname][name]['R2']; mae = results[tname][name]['MAE']
        ax.scatter(y, preds, color=color, alpha=0.75, s=45, edgecolors='white')
        lo,hi = min(y.min(),preds.min())-0.5, max(y.max(),preds.max())+0.5
        ax.plot([lo,hi],[lo,hi],'--',color='grey',alpha=0.5)
        style = 'bold' if name in PRIMARY else 'normal'
        ax.set_title(f'{name}\nR²={r2:.2f} MAE={mae:.2f}', fontsize=9, fontweight=style)
        if col==0:
            ax.set_ylabel('Predicted '+('Morning Feel' if 'feel' in tname else 'Productivity'))
        ax.set_xlabel('Actual')

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', bbox_inches='tight'); plt.show()

## 6. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
fig.suptitle('Model Comparison vs Naive Baseline (LOOCV, n=37)', fontsize=13, fontweight='bold')
model_names = list(models.keys())

for i, metric in enumerate(['MAE','RMSE','R2']):
    ax = axes[i]
    feel_vals = [results['morning_feel'][m][metric] for m in model_names]
    prod_vals  = [results['productivity'][m][metric]  for m in model_names]
    x = np.arange(len(model_names))
    ax.bar(x-0.2, feel_vals, 0.35, label='Morning Feel',
           color=['steelblue' if m in PRIMARY else 'cornflowerblue' for m in model_names], alpha=0.85)
    ax.bar(x+0.2, prod_vals,  0.35, label='Productivity',
           color=['coral' if m in PRIMARY else 'lightsalmon' for m in model_names], alpha=0.85)
    if metric == 'MAE':
        ax.axhline(base_mae_feel, color='steelblue', linestyle=':', alpha=0.7, label='Baseline Feel')
        ax.axhline(base_mae_prod,  color='coral',     linestyle=':', alpha=0.7, label='Baseline Prod')
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
    ax.set_title(metric); ax.legend(fontsize=7)
    ax.axvline(2.5, color='grey', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight'); plt.show()

## 7. SHAP Feature Importance

In [ ]:
def best_primary(target):
    return min(PRIMARY, key=lambda m: results[target][m]['MAE'])

best_feel = best_primary('morning_feel')
best_prod  = best_primary('productivity')
print(f'Best primary model — Morning Feel: {best_feel}')
print(f'Best primary model — Productivity:  {best_prod}')

fig, axes = plt.subplots(1, 2, figsize=(16,8))
fig.suptitle('SHAP Feature Importance (top 15, trained on full dataset)', fontsize=13, fontweight='bold')

for ax, (tname, y, best_name) in zip(axes, [
    ('morning_feel', y_feel, best_feel),
    ('productivity',  y_prod,  best_prod)
]):
    model = models[best_name]; model.fit(X_scaled, y)
    explainer = shap.LinearExplainer(model, X_scaled, feature_perturbation='interventional')
    sv = explainer.shap_values(X_scaled)
    mean_abs = np.abs(sv).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:15]
    ax.barh([FEAT_LABELS[o] for o in order[::-1]],
            mean_abs[order[::-1]],
            color='steelblue' if 'feel' in tname else 'coral', alpha=0.85)
    label = 'Morning Feel' if 'feel' in tname else 'Productivity'
    ax.set_title(f'{label}\n{best_name} (top 15 by mean |SHAP|)')
    ax.set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig('shap_importance.png', bbox_inches='tight'); plt.show()

## 8. SHAP Beeswarm — Feature Directions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,9))
fig.suptitle('SHAP Beeswarm — Direction & Magnitude of Feature Effects', fontsize=13, fontweight='bold')

for ax, (tname, y, best_name) in zip(axes, [
    ('morning_feel', y_feel, best_feel),
    ('productivity',  y_prod,  best_prod)
]):
    model = models[best_name]; model.fit(X_scaled, y)
    explainer = shap.LinearExplainer(model, X_scaled, feature_perturbation='interventional')
    sv = explainer.shap_values(X_scaled)
    mean_abs = np.abs(sv).mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:15]
    for j, fi in enumerate(order[::-1]):
        s = sv[:, fi]; fv = X_scaled[:, fi]
        norm = plt.Normalize(fv.min(), fv.max())
        colors_pt = plt.cm.RdBu_r(norm(fv))
        jitter = np.random.uniform(-0.15, 0.15, len(s))
        ax.scatter(s, np.full(len(s),j)+jitter, c=colors_pt, s=28, alpha=0.8)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([FEAT_LABELS[i] for i in order[::-1]], fontsize=8)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('SHAP value')
    label = 'Morning Feel' if 'feel' in tname else 'Productivity'
    ax.set_title(f'{label} — {best_name}\n← negative impact | positive impact →')

sm = plt.cm.ScalarMappable(cmap='RdBu_r'); sm.set_array([])
fig.colorbar(sm, ax=axes, label='Feature value (blue=low, red=high)', shrink=0.5)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', bbox_inches='tight'); plt.show()

## 9. Joint Optimization & Personalized Routine

In [ ]:
feel_p = predictions['morning_feel'][best_feel]
prod_p  = predictions['productivity'][best_prod]
fn = (feel_p - feel_p.min())/(feel_p.max()-feel_p.min())
pn = (prod_p  - prod_p.min())/(prod_p.max()-prod_p.min())
composite = (fn + pn)/2
ml_df['composite'] = composite

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('Dual-Target Joint Optimization', fontsize=13, fontweight='bold')

ax = axes[0]
sc = ax.scatter(feel_p, prod_p, c=composite, cmap='RdYlGn', s=80, edgecolors='grey', alpha=0.9)
plt.colorbar(sc, ax=ax, label='Composite score')
ax.set_xlabel('Predicted Morning Feel'); ax.set_ylabel('Predicted Productivity')
ax.set_title('Joint Prediction Space')

ax = axes[1]
ax.plot(ml_df['date'], composite, 'o-', color='teal', linewidth=2)
ax.fill_between(ml_df['date'], composite, alpha=0.2, color='teal')
best_idx = composite.argmax()
ax.axvline(ml_df['date'].iloc[best_idx], color='green', linestyle='--',
           label=f'Best: {ml_df["date"].iloc[best_idx].strftime("%b %d")}')
ax.set_ylabel('Composite Score (0–1)'); ax.set_title('Composite Score Over Time'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('joint_optimization.png', bbox_inches='tight'); plt.show()

# Optimal routine from top-third nights
thresh = np.percentile(composite, 67)
good = ml_df[ml_df['composite'] >= thresh]

def fh(h):
    if pd.isna(h): return 'N/A'
    return f"{int(h):02d}:{int((h%1)*60):02d}"

print('='*65)
print('PERSONALIZED OPTIMAL DAILY ROUTINE — Deniz Elbek')
print('Based on top-third nights by joint composite score')
print('='*65)
rows = [
    ('duration_h',         'Sleep Duration',       '{:.1f}h'),
    ('bedtime_hour',       'Bedtime',               None),
    ('wake_hour',          'Wake Time',             None),
    ('bedtime_after_sunset','Bedtime After Sunset', '{:.1f}h'),
    ('caffeine_mg',        'Daily Caffeine',        '{:.0f}mg'),
    ('last_caffeine_hour', 'Last Caffeine',         None),
    ('last_meal_hour',     'Last Meal',             None),
    ('alcohol_scale',      'Alcohol (0–5)',         '{:.0f}'),
    ('steps',              'Daily Steps',           '{:,.0f}'),
    ('exercise_min',       'Exercise',              '{:.0f} min'),
    ('nap_duration_min',   'Nap Duration',          '{:.0f} min'),
    ('temp_c',             'Optimal Sleep Temp',    '{:.1f}°C'),
]
for col, label, fmt in rows:
    gv = good[col].median(); av = ml_df[col].median()
    gs = fh(gv) if fmt is None else fmt.format(gv)
    as_ = fh(av) if fmt is None else fmt.format(av)
    print(f'{label:<25} Optimal: {gs:<12} Overall median: {as_}')

print('\n'+'='*65)
print('SUMMARY')
print('='*65)
print(f'Sleep window   : {fh(good["bedtime_hour"].median())} → {fh(good["wake_hour"].median())} (~{good["duration_h"].median():.1f}h)')
print(f'Caffeine       : ≤{good["caffeine_mg"].median():.0f}mg/day, last intake by {fh(good["last_caffeine_hour"].median())}')
print(f'Last meal      : by {fh(good["last_meal_hour"].median())}')
print(f'Alcohol        : {good["alcohol_scale"].median():.0f}/5 (minimise)')
print(f'Daily steps    : ≥{good["steps"].median():,.0f}')
print(f'Bedtime offset : within {good["bedtime_after_sunset"].median():.1f}h after sunset')

## 10. Summary & Limitations

### Key Findings
- A dual-target LOOCV pipeline successfully predicts morning feel rating and daily productivity from 24 personal sleep, lifestyle, and environmental features.
- Regularized linear models (Ridge/Lasso/ElasticNet) are the primary approach; tree-based models (Random Forest, XGBoost) serve as secondary comparisons.
- A naive mean baseline is included for honest model evaluation; any model beating it represents genuine predictive signal.
- SHAP analysis identifies the most personally impactful features for each outcome, forming the basis of a data-driven daily routine recommendation.
- Weather and circadian features (temperature, daylight hours, bedtime-after-sunset) add environmental context beyond physiological and lifestyle variables.

### Limitations
- **Sample size (n=37):** Small dataset; findings are personal and not statistically generalisable.
- **Self-reporting bias:** Morning feel and productivity ratings are subjective.
- **Environmental confound:** Roommate snoring elevates awake_min independently of lifestyle choices.
- **Weather data:** Weather API access was unavailable in the project environment. Climatologically realistic values for Tuzla, Istanbul were generated using NOAA historical norms and astronomical computation (astral library). Sunrise/sunset times are astronomically exact.
- **Missing Watch nights:** 7 nights excluded due to Apple Watch not being worn.
- **Causality:** All findings are correlational.